## Statistical Testing
Author: Angela Palacino

The purpose of this notebook is to perform statistical testing to verify if the number of cases is equally distributed among age groups and localities.
The statistical test permoformed is Chi-squared, the hypothesis test of independence of the observed frequencies in the contingency table. 
- The null hypothesis: Drug abuse cases are independent of age group and locality — no association.
- The alternative hypothesis: Drug abuse cases are dependent on age group and/or locality — there is an association.

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency

In [2]:
# Read the parquet file with the processed data
master_df = pd.read_parquet(
    "/home/ange/DS_projects/psycho_substance_abuse/data/processed/master_population_cases.parquet.gzip",
    engine="pyarrow",
)

In [3]:
master_df

,Localidad,age_group,CASOS,Población
0,Antonio Nariño,Adolescence,94.0,4620
1,Antonio Nariño,Adulthood,597.0,39247
2,Antonio Nariño,Elderly,85.0,14532
3,Antonio Nariño,First infancy,0.0,9181
4,Antonio Nariño,Infancy,0.0,4843
...,...,...,...,...
115,Usme,Adulthood,1101.0,183713
116,Usme,Elderly,73.0,51206
117,Usme,First infancy,0.0,44135
118,Usme,Infancy,17.0,35975


In [4]:
# Create the cases per 1000 habitants to have values according to the test (>5)
master_df["Cases per 1000"] = master_df["CASOS"] / master_df["Población"] * 1000

master_df.head()

,Localidad,age_group,CASOS,Población,Cases per 1000
0,Antonio Nariño,Adolescence,94.0,4620,20.346320
1,Antonio Nariño,Adulthood,597.0,39247,15.211354
2,Antonio Nariño,Elderly,85.0,14532,5.849160
3,Antonio Nariño,First infancy,0.0,9181,0.000000
4,Antonio Nariño,Infancy,0.0,4843,0.000000


To perfrom the statistical test is necessary to create a contingency table and remove any localities or age groups that have only 0 in their values.

In [13]:
# Create contingency table, sort values by age group, drop first infancy age group
chi_squared_df = master_df.pivot(
    index="age_group", columns="Localidad", values="Cases per 1000"
)

custom_dict = {
    "First Infancy": 0,
    "Infancy": 1,
    "Adolescence": 2,
    "Youth": 3,
    "Adulthood": 4,
    "Elderly": 5,
}
chi_squared_df.sort_values(
    by="age_group", key=lambda x: x.map(custom_dict), ascending=False, inplace=True
)
chi_squared_df = chi_squared_df.round(0)
chi_squared_df

Localidad,Antonio Nariño,Barrios Unidos,Bosa,Candelaria,Chapinero,Ciudad Bolívar,Engativá,Fontibón,Kennedy,Mártires,Puente Aranda,Rafael Uribe,San Cristóbal,Santa Fe,Suba,Sumapaz,Teusaquillo,Tunjuelito,Usaquén,Usme
age_group,,,,,,,,,,,,,,,,,,,,
Elderly,6.0,2.0,1.0,27.0,2.0,1.0,1.0,2.0,2.0,40.0,5.0,2.0,2.0,19.0,1.0,0.0,2.0,3.0,1.0,1.0
Adulthood,15.0,9.0,6.0,85.0,9.0,4.0,5.0,7.0,5.0,149.0,20.0,6.0,8.0,70.0,3.0,0.0,10.0,11.0,4.0,6.0
Youth,37.0,32.0,32.0,195.0,56.0,26.0,22.0,33.0,20.0,163.0,60.0,27.0,30.0,87.0,19.0,3.0,65.0,51.0,20.0,32.0
Adolescence,20.0,27.0,45.0,65.0,28.0,48.0,18.0,23.0,18.0,45.0,27.0,33.0,60.0,47.0,33.0,9.0,62.0,80.0,16.0,72.0
Infancy,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
First infancy,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [14]:
less_five = chi_squared_df[chi_squared_df > 5]
null_counts = less_five.isnull().sum()
print(f"There are {null_counts.sum()} values less than five.")

There are 63 values less than five.


In [15]:
chi_squared_df = chi_squared_df.drop("First infancy")
chi_squared_df

Localidad,Antonio Nariño,Barrios Unidos,Bosa,Candelaria,Chapinero,Ciudad Bolívar,Engativá,Fontibón,Kennedy,Mártires,Puente Aranda,Rafael Uribe,San Cristóbal,Santa Fe,Suba,Sumapaz,Teusaquillo,Tunjuelito,Usaquén,Usme
age_group,,,,,,,,,,,,,,,,,,,,
Elderly,6.0,2.0,1.0,27.0,2.0,1.0,1.0,2.0,2.0,40.0,5.0,2.0,2.0,19.0,1.0,0.0,2.0,3.0,1.0,1.0
Adulthood,15.0,9.0,6.0,85.0,9.0,4.0,5.0,7.0,5.0,149.0,20.0,6.0,8.0,70.0,3.0,0.0,10.0,11.0,4.0,6.0
Youth,37.0,32.0,32.0,195.0,56.0,26.0,22.0,33.0,20.0,163.0,60.0,27.0,30.0,87.0,19.0,3.0,65.0,51.0,20.0,32.0
Adolescence,20.0,27.0,45.0,65.0,28.0,48.0,18.0,23.0,18.0,45.0,27.0,33.0,60.0,47.0,33.0,9.0,62.0,80.0,16.0,72.0
Infancy,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
chi_squared_df_2 = chi_squared_df.drop("Infancy")
less_five_2 = chi_squared_df_2[chi_squared_df_2 > 5]
null_counts_2 = less_five_2.isnull().sum()
print(f"There are {null_counts_2.sum()} values less than five.")

There are 23 values less than five.


In [17]:
# Perform the Chi squared test on the contingency table with infancy age group
chi2, p, dof, expected = chi2_contingency(chi_squared_df)
print(f"P-value: {p}")

P-value: 1.2851623926172651e-64


In [18]:
# Perform the Chi squared test on the contingency table without infancy age group (17 zero values)
chi2, p, dof, expected = chi2_contingency(chi_squared_df_2)
print(f"P-value: {p}")

P-value: 4.43822869401884e-71


The obtained p-value is lower than 0.05 which means it is safe to reject the null hypothesis that drug abuse cases are independent of age group and locality. Furthermore, the p-value is quite small to say there is a significant association between age group, locality, and drug abuse cases.